# SBP data acquisition

Simulation writes zero traces and opens no instruments. Complete calibration before a real scan. [Hardware details](README.md#calibration-and-data-acquisition).

In [ ]:
from pathlib import Path
from src.acquisition import (
    MeasurementConfig, create_demo_config, load_calibration,
    plan_scan, acquire, summarize_run,
)

SIMULATE = True
VNA_TESTMODE = SIMULATE
CONFIG_FILE = (create_demo_config("outputs/acquisition") if SIMULATE
               else Path("config/measurement_config.json"))
config = MeasurementConfig.from_file(CONFIG_FILE)

## 1. Settings and calibration

Review motor limits, VNA sweep and output directory. Units: metres and Hz.

In [ ]:
calibration = load_calibration(config)
print(calibration.summary())
print("Motor limits:", config.stepper.min_position_steps, config.stepper.max_position_steps)
print(f"VNA: S{config.vna.port_a}{config.vna.port_b}, channel {config.vna.channel}, "
      f"{config.vna.start_freq_Hz/1e9:g}–{config.vna.stop_freq_Hz/1e9:g} GHz, "
      f"{config.vna.points} points, {config.vna.averages} averages")
print("Simulated motor:", SIMULATE, "VNA zero-trace mode:", VNA_TESTMODE)

## 2. Preview scan

No hardware connection.

In [ ]:
targets = plan_scan(config)
print("Planned points:", len(targets))
print("thread  point  position [m]  target [steps]")
for sub, index, _, position, steps in targets[:20]:
    print(f"{sub.index:6d} {index:6d} {position:13.6f} {steps:15d}")

## 3. Acquire

Interrupt aborts and releases instruments. After emergency STOP, re-establish motor position.

In [ ]:
run_directory = acquire(
    config, simulate=SIMULATE, vna_testmode=VNA_TESTMODE,
    config_path=CONFIG_FILE,
)
print("Saved run:", run_directory)

## 4. Saved run

Inspect traces, positions and manifest. The HDF5 attributes identify the actual S-parameter.

In [ ]:
summarize_run(run_directory)